# CSI4142 - Assignment 2: Data Cleaning
**Group Number:** 99
**Group Members:**
* Suryadev Andotra [300006733]
* Ryan Jiayan Guo [300294370]

**Work Split:**
* **Suryadev:** Dataset 1 & 2 selection, Notebook setup, Validity Tests 1-5 (Data Type, Range, Format, Consistency, Uniqueness), and Imputation Test 1 (Regression).
* **Ryan:** Validity Tests 6-8 (Presence, Length, Look-up) and Imputation Tests 2 & 3 (Default Value, Similarity-based), including quantitative evaluations.

In [3]:
import pandas as pd
import numpy as np
import random

url_hotel = "https://raw.githubusercontent.com/Sandotra/CSI4142-Assignment2-2026/refs/heads/main/hotel_bookings.csv"
url_air = "https://raw.githubusercontent.com/Sandotra/CSI4142-Assignment2-2026/refs/heads/main/c4_epa_air_quality.csv"

# Load the datasets
df_hotel = pd.read_csv(url_hotel)
df_air = pd.read_csv(url_air)

print(f"Dataset 1 (Hotel) Shape: {df_hotel.shape}")
print(f"Dataset 2 (Air Quality) Shape: {df_air.shape}")

Dataset 1 (Hotel) Shape: (119390, 32)
Dataset 2 (Air Quality) Shape: (260, 10)


### Dataset 1: Hotel Booking Demand (Validity Checking)
* **Author:** Jesse Mostipak (Kaggle)
* **Purpose:** Real, non-synthetic data aggregated to predict hotel booking cancellations and analyze booking behaviors.
* **Shape:** ~119,390 rows, 32 columns.
* **Key Features:** `adults` (Numerical - Number of adults), `adr` (Numerical - Average Daily Rate), `reservation_status_date` (Categorical/Date - Date of status), `is_canceled` (Categorical/Binary - 1 if canceled, 0 if not).

### Dataset 2: Air Quality Data (Imputation)
* **Author:** Yakhyojon (Kaggle)
* **Purpose:** Real, non-synthetic data collected from sensors to track atmospheric pollutants and weather conditions.
* **Shape:** ~260 rows, 10 columns.
* **Key Features:** `Temperature` (Numerical), `Humidity` (Numerical), `PM2.5` (Numerical - Fine particulate matter).

### 1. Data Type Errors
**Description:** In this test, we verify that a column meant for numeric values contains only numbers. A data type error occurs when a string or other incompatible data type is introduced into a numerical column.

In [ ]:
# Original dataset before modifying
df_hotel_dirty = df_hotel.copy()

df_hotel_dirty['adults'] = df_hotel_dirty['adults'].astype(object)

# Introduce 5% error into the 'adults' column (Numerical)
# Replace 5% of the integer values with the string "Invalid_String"
np.random.seed(42)
error_indices = np.random.choice(df_hotel_dirty.index, size=int(0.05 * len(df_hotel_dirty)), replace=False)

df_hotel_dirty.loc[error_indices, 'adults'] = "Invalid_String"

print(f"Introduced data type errors at {len(error_indices)} indices in the 'adults' column.")

Introduced data type errors at 5969 indices in the 'adults' column.


In [ ]:
# Code that will automatically find the error
def check_datatype_errors(df, column_name):
    """Flags rows where the value cannot be converted to a numeric type."""
    # Turns invalid strings into NaN
    numeric_conversion = pd.to_numeric(df[column_name], errors='coerce')

    # Find rows that are NaN after conversion, but weren't NaN originally
    error_mask = numeric_conversion.isna() & df[column_name].notna()
    return df[error_mask]

# Run the checker
datatype_errors_found = check_datatype_errors(df_hotel_dirty, 'adults')
print(f"Checker found {len(datatype_errors_found)} data type errors.")

Checker found 5969 data type errors.


In [ ]:
# Qualitative results
print("Qualitative Results: Below are 3 examples showing the original valid data alongside the corrupted data detected by the checker.\n")

# 3 examples
sample_errors = datatype_errors_found.head(3).index

for idx in sample_errors:
    original_val = df_hotel.loc[idx, 'adults']
    dirty_val = df_hotel_dirty.loc[idx, 'adults']
    print(f"Row {idx}: Original Value = {original_val} (Type: {type(original_val).__name__}) | Corrupted Value = '{dirty_val}' (Type: {type(dirty_val).__name__})")

Qualitative Results: Below are 3 examples showing the original valid data alongside the corrupted data detected by the checker.

Row 44: Original Value = 2 (Type: int64) | Corrupted Value = 'Invalid_String' (Type: str)
Row 62: Original Value = 2 (Type: int64) | Corrupted Value = 'Invalid_String' (Type: str)
Row 87: Original Value = 2 (Type: int64) | Corrupted Value = 'Invalid_String' (Type: str)


### 2. Range Errors
**Description:** In this test, we verify that a numerical attribute falls within a logical minimum and maximum range. A range error occurs when a value exceeds these bounds. For example, a hotel's Average Daily Rate (`adr`) cannot realistically be negative.

In [ ]:
# Introduce range errors by setting 5% of 'adr' values to a negative price (-150.0).
np.random.seed(42)
error_indices_range = np.random.choice(df_hotel_dirty.index, size=int(0.05 * len(df_hotel_dirty)), replace=False)

df_hotel_dirty.loc[error_indices_range, 'adr'] = -150.0
print(f"Introduced range errors (negative adr) at {len(error_indices_range)} indices.")

Introduced range errors (negative adr) at 5969 indices.


In [ ]:
# Code that will automatically find the error
def check_range_errors(df, column_name, min_val=0):
    """Flags rows where the value is below the minimum allowed value."""
    # Use coerce to ignore the Data Type errors introduced in Test 1
    numeric_col = pd.to_numeric(df[column_name], errors='coerce')
    error_mask = numeric_col < min_val
    return df[error_mask]

range_errors_found = check_range_errors(df_hotel_dirty, 'adr', min_val=0)
print(f"Checker found {len(range_errors_found)} range errors.")

Checker found 5970 range errors.


In [ ]:
# Qualitative results
print("Qualitative Results: 3 examples showing the original valid adr alongside the corrupted negative adr.\n")
sample_range_errors = range_errors_found.head(3).index

for idx in sample_range_errors:
    original_val = df_hotel.loc[idx, 'adr']
    dirty_val = df_hotel_dirty.loc[idx, 'adr']
    print(f"Row {idx}: Original 'adr' = {original_val} | Corrupted 'adr' = {dirty_val}")

Qualitative Results: 3 examples showing the original valid adr alongside the corrupted negative adr.

Row 44: Original 'adr' = 110.7 | Corrupted 'adr' = -150.0
Row 62: Original 'adr' = 133.0 | Corrupted 'adr' = -150.0
Row 87: Original 'adr' = 108.73 | Corrupted 'adr' = -150.0


### 3. Format Errors
**Description:** In this test, we verify that strings representing structured data follow a strict formatting pattern. We will check the `reservation_status_date` which should strictly follow the `YYYY-MM-DD` format.

In [ ]:
# Introduce format errors by changing 5% of dates to a DD/MM/YYYY format
np.random.seed(42)
error_indices_format = np.random.choice(df_hotel_dirty.index, size=int(0.05 * len(df_hotel_dirty)), replace=False)

# Reformat the selected dates
df_hotel_dirty.loc[error_indices_format, 'reservation_status_date'] = df_hotel_dirty.loc[error_indices_format, 'reservation_status_date'].apply(
    lambda x: '/'.join(str(x).split('-')[::-1]) if pd.notnull(x) else x
)
print(f"Introduced format errors (changed to DD/MM/YYYY) at {len(error_indices_format)} indices.")

Introduced format errors (changed to DD/MM/YYYY) at 5969 indices.


In [ ]:
# Code that will automatically find the error
import re

def check_format_errors(df, column_name, pattern):
    """Flags rows where the string does not match the expected regex pattern."""
    error_mask = ~df[column_name].astype(str).str.match(pattern)
    return df[error_mask]

# YYYY-MM-DD pattern
date_pattern = r'^\d{4}-\d{2}-\d{2}$'
format_errors_found = check_format_errors(df_hotel_dirty, 'reservation_status_date', date_pattern)
print(f"Checker found {len(format_errors_found)} format errors.")

Checker found 5969 format errors.


In [ ]:
# Qualitative results
print("Qualitative Results: 3 examples of broken date formats detected by the checker.\n")
sample_format_errors = format_errors_found.head(3).index

for idx in sample_format_errors:
    original_val = df_hotel.loc[idx, 'reservation_status_date']
    dirty_val = df_hotel_dirty.loc[idx, 'reservation_status_date']
    print(f"Row {idx}: Original Date = {original_val} | Corrupted Format = {dirty_val}")

Qualitative Results: 3 examples of broken date formats detected by the checker.

Row 44: Original Date = 2015-07-09 | Corrupted Format = 09/07/2015
Row 62: Original Date = 2015-07-05 | Corrupted Format = 05/07/2015
Row 87: Original Date = 2015-04-15 | Corrupted Format = 15/04/2015


### 4. Consistency Errors
**Description:** In this test, we verify logical consistency between two attributes. For example, if a booking is officially marked as canceled (`is_canceled == 1`), the `reservation_status` cannot logically be `Check-Out`.

In [ ]:
# Find canceled rows
canceled_indices = df_hotel_dirty[df_hotel_dirty['is_canceled'] == 1].index

# Corrupt 5%
np.random.seed(42)
error_indices_consist = np.random.choice(canceled_indices, size=int(0.05 * len(canceled_indices)), replace=False)

# Introduce logic contradiction
df_hotel_dirty.loc[error_indices_consist, 'reservation_status'] = 'Check-Out'
print(f"Introduced consistency errors at {len(error_indices_consist)} indices (Canceled bookings marked as Check-Out).")

Introduced consistency errors at 2211 indices (Canceled bookings marked as Check-Out).


In [ ]:
# Code that will automatically find the error
def check_consistency_errors(df):
    """Flags rows where is_canceled is 1 but reservation_status is 'Check-Out'."""
    error_mask = (df['is_canceled'] == 1) & (df['reservation_status'] == 'Check-Out')
    return df[error_mask]

consistency_errors_found = check_consistency_errors(df_hotel_dirty)
print(f"Checker found {len(consistency_errors_found)} consistency errors.")

Checker found 2211 consistency errors.


In [ ]:
# Qualitative results
print("Qualitative Results: 3 examples of logical contradictions detected by the checker.\n")
sample_consist_errors = consistency_errors_found.head(3).index

for idx in sample_consist_errors:
    orig_status = df_hotel.loc[idx, 'reservation_status']
    dirty_status = df_hotel_dirty.loc[idx, 'reservation_status']
    is_canc = df_hotel_dirty.loc[idx, 'is_canceled']
    print(f"Row {idx}: is_canceled = {is_canc} | Original Status = {orig_status} | Corrupted Status = {dirty_status}")

Qualitative Results: 3 examples of logical contradictions detected by the checker.

Row 38: is_canceled = 1 | Original Status = Canceled | Corrupted Status = Check-Out
Row 233: is_canceled = 1 | Original Status = Canceled | Corrupted Status = Check-Out
Row 365: is_canceled = 1 | Original Status = Canceled | Corrupted Status = Check-Out


### 5. Uniqueness Errors
**Description:** In this test, we verify that rows representing distinct entities do not have identical duplicates. We will intentionally duplicate 5% of the dataset to simulate uniqueness errors (accidental duplicate entries).

In [ ]:
# Add a unique 'booking_id'
if 'booking_id' not in df_hotel_dirty.columns:
    df_hotel_dirty.insert(0, 'booking_id', range(1, 1 + len(df_hotel_dirty)))

# Duplicate 5% of the current rows and append them to the dataset
num_duplicates = int(0.05 * len(df_hotel_dirty))
duplicate_rows = df_hotel_dirty.sample(n=num_duplicates, random_state=42)

df_hotel_dirty = pd.concat([df_hotel_dirty, duplicate_rows], ignore_index=True)
print(f"Appended {num_duplicates} identical duplicate rows to introduce uniqueness errors.")

Appended 5969 identical duplicate rows to introduce uniqueness errors.


In [ ]:
# Code that will automatically find the error
def check_uniqueness_errors(df):
    """Flags rows that are exact duplicates of another row."""
    # keep=False marks all duplicates (the original and the copies)
    error_mask = df.duplicated(keep=False)
    return df[error_mask]

uniqueness_errors_found = check_uniqueness_errors(df_hotel_dirty)
print(f"Checker found {len(uniqueness_errors_found)} rows involved in duplication.")

Checker found 11938 rows involved in duplication.


In [ ]:
# Qualitative results
print("Qualitative Results: Below is an example showing identical duplicated rows (including our generated booking_id) detected by the checker.\n")

# Show the copies of a specific duplicate
if not uniqueness_errors_found.empty:
    dup_example_id = uniqueness_errors_found.iloc[0]['booking_id']

    specific_duplicates = uniqueness_errors_found[uniqueness_errors_found['booking_id'] == dup_example_id].head(2)

    display_df = specific_duplicates[['booking_id', 'hotel', 'lead_time', 'adults', 'adr']].reset_index(names="Row_Number")

    # Print
    print(display_df.to_string(index=False))
else:
    print("No duplicates found.")

Qualitative Results: Below is an example showing identical duplicated rows (including our generated booking_id) detected by the checker.

 Row_Number  booking_id        hotel  lead_time         adults    adr
         44          45 Resort Hotel        107 Invalid_String -150.0
     119841          45 Resort Hotel        107 Invalid_String -150.0


### 6. Presence Errors
Presence Errors check if columns and rows have SOMETHING in them. If not, error. I introduced 10 percent error for testing. Results at the end.

In [ ]:
# Add 10% error

np.random.seed(42)
df_hotel_dirty = df_hotel.copy()
presence_indices = np.random.choice(
    df_hotel_dirty.index,
    size=int(0.1 * len(df_hotel_dirty)),
    replace=False
)

df_hotel_dirty.loc[presence_indices, 'children'] = np.nan

print(f"Introduced {len(presence_indices)} presence errors.")

Introduced 11939 presence errors.


In [ ]:
def check_presence_errors(df, column_name):
    return df[df[column_name].isna()]

presence_errors_found = check_presence_errors(df_hotel_dirty, 'children')
print(f"Checker found {len(presence_errors_found)} presence errors.")
presence_errors_found['children'].head()

Checker found 11943 presence errors.


,children
35,NaN
39,NaN
44,NaN
62,NaN
70,NaN


### Qualitative Results:

The presence checker identified 11,943 missing values in the 'children' column.

Below are 3 examples of detected presence errors:

Row 35: children = NaN  
Row 39: children = NaN  
Row 44: children = NaN  

There are many presence errors including ones not in the children column

### 7. Length Errors

Length errors occur when a string does not match the expected length.
For example, country codes should have exactly 3 characters. CANADA = CAN

In [ ]:
np.random.seed(42)
df_hotel_dirty = df_hotel.copy()

length_indices = np.random.choice(
    df_hotel_dirty.index,
    size=int(0.07 * len(df_hotel_dirty)),
    replace=False
)

df_hotel_dirty.loc[length_indices, 'country'] = \
    df_hotel_dirty.loc[length_indices, 'country'].str[:2]

print(f"Introduced {len(length_indices)} length errors in 'country'.")

Introduced 8357 length errors in 'country'.


In [ ]:
def check_length_errors(df, column_name, expected_length):
    return df[df[column_name].astype(str).str.len() != expected_length]

length_errors_found = check_length_errors(df_hotel_dirty, 'country', 3)

print(f"Checker found {len(length_errors_found)} length errors.")
length_errors_found[['country', 'hotel', 'stays_in_weekend_nights']].head(3)

Checker found 9515 length errors.


,country,hotel,stays_in_weekend_nights
35,PR,Resort Hotel,1
39,RO,Resort Hotel,2
44,PR,Resort Hotel,2


### Qualitative Results:

The presence checker identified 9515 missing values in the 'country' column.

Below are 3 examples of detected presence errors:

Row 35: country = PR instead of PRT

Row 39: country = RO instead of ROM

Row 44: country - PR instead of PRT

There are many presence errors including ones not in the country column

### 8. Look-up Errors

Description:
Look-up errors occur when a value is not part of an allowed predefined list.
For example, reservation_status should only contain valid status values.

In [ ]:
np.random.seed(42)
df_hotel_dirty = df_hotel.copy()

lookup_indices = np.random.choice(
    df_hotel_dirty.index,
    size=int(0.067 * len(df_hotel_dirty)),
    replace=False
)

df_hotel_dirty.loc[lookup_indices, 'reservation_status'] = "abc"

print(f"Introduced {len(lookup_indices)} look-up errors.")

Introduced 7999 look-up errors.


In [ ]:
valid_status = ["Canceled", "Check-Out", "No-Show"]

def check_lookup_errors(df, column_name, valid_values):
    return df[~df[column_name].isin(valid_values)]

lookup_errors_found = check_lookup_errors(
    df_hotel_dirty,
    'reservation_status',
    valid_status
)

print(f"Checker found {len(lookup_errors_found)} look-up errors.")
lookup_errors_found['reservation_status'].head(3)

Checker found 7999 look-up errors.


,reservation_status
39,abc
44,abc
62,abc


### Qualitative Results:

The presence checker identified 7999 incorrect reservation_statuses

Below are 3 examples of detected presence errors:

Row 39 : reservation status = 'abc'

Row 44 : reservation status = 'abc'

Row 62 : reservation status = 'abc'


### Imputation Test 1: Regression Imputation (Bivariate)
**Description:** In this test, we use Bivariate Regression Imputation. We will predict missing values of the `aqi` (Air Quality Index) based on its strong linear correlation with `arithmetic_mean` (Pollutant concentration). We are simulating an **MCAR** (Missing Completely At Random) scenario.

In [ ]:
# Simulating MCAR by removing 5% of 'aqi' values randomly
df_air_missing = df_air.copy()

np.random.seed(42)
missing_indices = np.random.choice(df_air_missing.index, size=int(0.05 * len(df_air_missing)), replace=False)
df_air_missing.loc[missing_indices, 'aqi'] = np.nan

print(f"Introduced MCAR missing data by removing 'aqi' values at {len(missing_indices)} indices.")

Introduced MCAR missing data by removing 'aqi' values at 13 indices.


In [ ]:
from sklearn.linear_model import LinearRegression

def regression_impute(df, feature_col, target_col):
    df_imputed = df.copy()

    train_data = df_imputed[df_imputed[target_col].notna()]
    missing_data = df_imputed[df_imputed[target_col].isna()]

    # Train the model
    X_train = train_data[[feature_col]]
    y_train = train_data[target_col]
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict the missing values and round to nearest whole number
    X_missing = missing_data[[feature_col]]
    predictions = np.round(model.predict(X_missing))

    # Fill the gaps
    df_imputed.loc[missing_data.index, target_col] = predictions
    return df_imputed

df_air_imputed = regression_impute(df_air_missing, 'arithmetic_mean', 'aqi')
print("Successfully imputed missing 'aqi' values using Regression.")

Successfully imputed missing 'aqi' values using Regression.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Quantitative Evaluation: Compare imputed predictions against the original known values
original_values = df_air.loc[missing_indices, 'aqi']
imputed_values = df_air_imputed.loc[missing_indices, 'aqi']

mae = mean_absolute_error(original_values, imputed_values)
rmse = np.sqrt(mean_squared_error(original_values, imputed_values))

print("Quantitative Evaluation of Regression Imputation:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("\nSample Before/After Comparison:")
comparison_df = pd.DataFrame({'Original AQI': original_values, 'Imputed Predicted AQI': imputed_values}).head(5)
print(comparison_df)

Quantitative Evaluation of Regression Imputation:
Mean Absolute Error (MAE): 0.6154
Root Mean Squared Error (RMSE): 0.9608

Sample Before/After Comparison:
     Original AQI  Imputed Predicted AQI
30              2                    2.0
181             2                    2.0
223             2                    2.0
185             2                    1.0
211             3                    3.0


### Imputation Test 2 : Default Value Imputation (Median)

In this test, we applied Default Value Imputation to handle missing values in the arithmetic_mean attribute. After simulating missing values under an MCAR mechanism, we replaced each missing entry with a single default statistic computed from the observed data, which was the median of the attribute.

In [4]:
df_air_missing = df_air.copy()

df_air_missing['arithmetic_mean'] = pd.to_numeric(df_air_missing['arithmetic_mean'], errors='coerce')

available_indices = df_air_missing.index[df_air_missing['arithmetic_mean'].notna()]

num_missing = int(0.07 * len(available_indices))
np.random.seed(42)
missing_indices = np.random.choice(available_indices, size=num_missing, replace=False)
df_air_missing.loc[missing_indices, 'arithmetic_mean'] = np.nan

print(f"Introduced {len(missing_indices)} MCAR missing values in 'arithmetic_mean'.")

Introduced 18 MCAR missing values in 'arithmetic_mean'.


In [5]:
df_air_imputed = df_air_missing.copy()

median_value = df_air_imputed['arithmetic_mean'].median()
df_air_imputed['arithmetic_mean'] = df_air_imputed['arithmetic_mean'].fillna(median_value)

valid_indices = [i for i in missing_indices if pd.notna(df_air.loc[i, 'arithmetic_mean'])]

if valid_indices:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    import numpy as np

    original_values = df_air.loc[valid_indices, 'arithmetic_mean']
    imputed_values = df_air_imputed.loc[valid_indices, 'arithmetic_mean']

    mae = mean_absolute_error(original_values, imputed_values)
    rmse = np.sqrt(mean_squared_error(original_values, imputed_values))

    print("Default Imputation Evaluation:")
    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}")

    comparison_df = pd.DataFrame({'Original': original_values, 'Imputed': imputed_values}).head(5)
    print(comparison_df)
else:
    print("No valid values to compute MSE/MAE.")

Default Imputation Evaluation:
MAE: 0.1705, RMSE: 0.3049
     Original   Imputed
30   0.184211  0.281579
181  0.200000  0.281579
223  0.183333  0.281579
185  0.142105  0.281579
211  0.215789  0.281579


In this test, we simulate 10% missing values in the numeric column 'arithmetic_mean' using an MCAR scenario. Missing values are replaced with the column median. Evaluation is performed using MAE and RMSE by comparing the imputed values to the original data at positions that were originally numeric.

### Imputation Test 3 : Similarity-Based Imputation (KNN)

In this test, we applied a Similarity-Based Imputation method using the K-Nearest Neighbors (KNN) algorithm to handle missing values in the arithmetic_mean attribute.

For each missing value, the algorithm identifies the five most similar observations based on other numeric attributes and imputes the value using the average of those neighbors.

This is a multivariate approach, as it relies on multiple attributes to compute similarity between records. The performance of the method was evaluated using MAE and RMSE by comparing the imputed values to the original values.

In [ ]:
df_air_missing = df_air.copy()

# Ensure values are present
df_air_missing['arithmetic_mean'] = pd.to_numeric(df_air_missing['arithmetic_mean'], errors='coerce')

available_indices = df_air_missing.index[df_air_missing['arithmetic_mean'].notna()]

# Introduce 10% missing values
num_missing = int(0.13 * len(available_indices))
np.random.seed(42)
missing_indices = np.random.choice(available_indices, size=num_missing, replace=False)
df_air_missing.loc[missing_indices, 'arithmetic_mean'] = np.nan

print(f"Introduced {len(missing_indices)} MCAR missing values in 'arithmetic_mean'.")

Introduced 26 MCAR missing values in 'arithmetic_mean'.


In [ ]:
from sklearn.impute import KNNImputer
import numpy as np
import pandas as pd

df_air_imputed = df_air_missing.copy()

# MAke sure columns are numeric
numeric_cols = df_air_imputed.select_dtypes(include='number').columns
numeric_cols = [col for col in numeric_cols if df_air_imputed[col].notna().any()]


imputer = KNNImputer(n_neighbors=5)

imputed_array = imputer.fit_transform(df_air_imputed[numeric_cols])
df_air_imputed[numeric_cols] = pd.DataFrame(imputed_array, columns=numeric_cols, index=df_air_imputed.index)
valid_indices = [i for i in missing_indices if pd.notna(df_air.loc[i, 'arithmetic_mean'])]

if valid_indices:
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    original_values = df_air.loc[valid_indices, 'arithmetic_mean']
    imputed_values = df_air_imputed.loc[valid_indices, 'arithmetic_mean']

    mae = mean_absolute_error(original_values, imputed_values)
    rmse = np.sqrt(mean_squared_error(original_values, imputed_values))

    print("Similarity-Based (KNN) Imputation Evaluation:")
    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}")

    comparison_df = pd.DataFrame({'Original': original_values, 'Imputed': imputed_values}).head(5)
    print(comparison_df)
else:
    print("No valid values to compute MSE/MAE.")

Similarity-Based (KNN) Imputation Evaluation:
MAE: 0.1066, RMSE: 0.1567
     Original   Imputed
30   0.184211  0.274094
181  0.200000  0.201053
223  0.183333  0.246035
185  0.142105  0.247368
211  0.215789  0.203684


### References

**Datasets:**
1. Mostipak, J. (n.d.). *Hotel Booking Demand*. Kaggle. Retrieved from https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
2. Yakhyojon. (n.d.). *Air Quality Data*. Kaggle. Retrieved from https://www.kaggle.com/datasets/yakhyojon/air-quality-data

**Documentation & Libraries:**
3. Pandas Development Team. (2024). *pandas documentation* (Version 2.0+). Retrieved from https://pandas.pydata.org/docs/ (Used for `pd.to_numeric`, `duplicated`, and dataframe manipulation).
4. Scikit-learn Developers. (2024). *scikit-learn documentation*. Retrieved from https://scikit-learn.org/stable/ (Used for `LinearRegression`, `mean_absolute_error`, and `mean_squared_error`).